

<img src="NB_images\portada.png" style="width:750px" align="center">

<h1><center>Basic Python for Geosciences</center></h1>

<h2><center>Session 05 Extra SEGY files and digital processing </center></h2>



<h3>Course creators and instructors</h3>  

German Ocampo (Exploration), Manuel David Soto (Technical Direction), and Giorgio De Paola (TechLab)

<h3>Collaborators</h3>  

Rosa Vega (Technical Direction), and Gaurav Sharma (Production Engineering & Operational Subsurface)


<h3>Table of contents</h3>



* [1 Reading segy with ObsPy](#title82)
    * [1.1 Reading segy files](#title821)
    * [1.2 Reading the trace header](#title822)
    * [1.3 Reading the segy data](#title823)
    * [1.4 Writing a segyf file](#title824)
* [2 Digital processing](#title83)
    * [2.1 Fast fourier transform](#title831)
    * [2.2 Periodogram](#title832)
    * [2.3 Wavelet generation in python](#title833)
    * [2.4 Convolution](#title834)
    * [2.5 Correlation](#title835)
    * [2.6 Filter design](#title836)

    
    

    
    
    
    
    
    



<a  id="title82"></a> 
<h1>1. Reading segy with ObsPy</h1> 

<img src="nb_images/obspy.png" style="width: 600px;"/> 

ObsPy is an open-source project dedicated to provide a Python framework for processing seismological data. It provides parsers for common file formats, clients to access data centers and seismological signal processing routines which allow the manipulation of seismological time series.

The goal of the ObsPy project is to facilitate rapid application development for seismology.

In seismic , segy file is the most common data file format used for data processing and interpretation.   
  
    
The general structure of a SEGY file is:

<img src="nb_images/segy_structure.jpg" style="width: 600px;"/> 

For segy data manipulation, we recommend to import the following modules.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import os

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from obspy import read, Trace, Stream, UTCDateTime
from obspy.core import AttribDict
from obspy.io.segy.segy import SEGYTraceHeader, SEGYBinaryFileHeader
from obspy.io.segy.core import _read_segy


<a  id="title821"></a> 
<h2>1.1 Reading segy files    </h2> 

The segy is read in a stream python object. from there, the seismic data and the headers can be extracted

In [ ]:
filename="./input/Data.sgy"


#Obspy reads a stream of the segy file
st = _read_segy(filename,textual_header_encoding="EBCDIC",unpack_trace_headers=True)


#Get the number of samples
l1=st[0].stats.npts
print("The number of samples are            :",l1)

#Get number of traces
l2=len(st)  
print("The number of traces are             :",l2)

#Get the sampling rate
fs=st[0].stats.sampling_rate
print("The number of samples by second are  :", fs)

#Get the time sampling
delta=st[0].stats.delta
print("The time sampling is (s)             :", delta)


<a  id="title822"></a> 
<h2>1.2 Reading the trace header    </h2>

The headers are read in a dictionary, the keys of the dicitionary are based in the segy standard V1 

In [ ]:
#Printing trace header dictionary available
print("Trace headers available for the first trace :",st[0].stats.segy)

The format of the trace header data is read in int type

In [ ]:
#Extract information from headers, in this case we are going to extract the receiver, source coordinate
# and also the offset

x=[]
y=[]
offset=[]
for i in range(0,l2):
    x.append(st[i].stats.segy.trace_header["group_coordinate_x"])
    y.append(st[i].stats.segy.trace_header["group_coordinate_y"])
    offset.append(st[i].stats.segy.trace_header["distance_from_center_of_the_source_point_to_the_center_of_the_receiver_group"])
   
    
#Plotting headers
fig=plt.figure(figsize=(20,10))
plt.subplot(2,1,1)
plt.plot(x,y)
plt.title("Coordinates")
plt.xlabel("Easting")
plt.ylabel("Northing")
plt.grid(True)

plt.subplot(2,1,2)
plt.plot(x,offset)
plt.title("Offset")
plt.xlabel("Easting")
plt.ylabel("Northing")
plt.grid(True)    
    

<a  id="title823"></a> 
<h2>1.3 Reading the segy data    </h2>

In [ ]:
#Reading and plotting seismic data

data=np.zeros((l1,l2))

for i in range(0,l2):
    data[:,i]=st[i].data


fig=plt.figure(figsize=(20,10))
plt.subplot(1,1,1)
#plt.imshow(data,cmap='Greys_r',interpolation="nearest",extent=(1,l2+1,fs*l1,0),vmin=-10000,vmax=10000,aspect="auto")
plt.imshow(data,cmap='seismic',interpolation="nearest",extent=(1,l2+1,fs*l1,0),vmin=-10000,vmax=10000,aspect="auto")
plt.xlabel("Trace")
plt.ylabel("Time(s)")
plt.title("Segy file")
plt.colorbar()

To extract the data from the  trace n, just need to select the column n-1 of the data numpy array 

In [ ]:
#Selection of single trace

signal=data[:,301]

fig=plt.figure(figsize=(20,10))
plt.plot(signal)
plt.title("Trace "+str(300))
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.grid(True)

<a  id="title824"></a> 
<h2>1.4 Writing a segy file    </h2>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from obspy import read, Trace, Stream, UTCDateTime
from obspy.core import AttribDict
from obspy.io.segy.segy import SEGYTraceHeader, SEGYBinaryFileHeader
from obspy.io.segy.core import _read_segy

stream = Stream()

for i in range(0,l2):

    #Get the data from the column i to a list
    lista=data[:,i]
    data1 = np.require(lista, dtype=np.float32)  #Change to float32
    trace = Trace(data=data1) 


    trace.stats.delta = 0.02/10
    trace.stats.starttime = UTCDateTime(2011,11,11,11,11,11)

    # If you want to set some additional attributes in the trace header,
    # add one and only set the attributes you want to be set. Otherwise the
    # header will be created for you with default values.
    if not hasattr(trace.stats, 'segy.trace_header'):
        trace.stats.segy = {}
    trace.stats.segy.trace_header = SEGYTraceHeader()
    trace.stats.segy.trace_header.trace_sequence_number_within_line = i + 1
    trace.stats.segy.trace_header.receiver_group_elevation = 444

    # Add trace to stream
    stream.append(trace)

# A SEGY file has file wide headers. This can be attached to the stream
# object.  If these are not set, they will be autocreated with default
# values.
stream.stats = AttribDict()
stream.stats.textual_file_header = 'Textual Header!'
stream.stats.binary_file_header = SEGYBinaryFileHeader()
stream.stats.binary_file_header.trace_sorting_code = 5


filename_out="./output/output.sgy"
stream.write(filename_out, format="SEGY", data_encoding=1,byteorder=">")



<a  id="title83"></a> 
<h1> 2. Signal processing   </h1> 
    
    
https://docs.scipy.org/doc/scipy/reference/signal.html    

<a  id="title831"></a> 
<h2>2.1 Fast fourier transform    </h2> 

A fast Fourier transform (FFT) is an algorithm that computes the discrete Fourier transform (DFT) of a sequence, or its inverse (IDFT). Fourier analysis converts a signal from its original domain (often time or space) to a representation in the frequency domain and vice versa. The DFT is obtained by decomposing a sequence of values into components of different frequencies.[1] This operation is useful in many fields, but computing it directly from the definition is often too slow to be practical. An FFT rapidly computes such transformations by factorizing the DFT matrix into a product of sparse (mostly zero) factors

https://en.wikipedia.org/wiki/Fast_Fourier_transform

In [ ]:
#FFT calculation


spectrum=np.fft.fft(signal)
freq = np.fft.fftfreq(len(signal),d=1/fs)

fig=plt.figure(figsize=(20,5))


plt.plot(freq,np.real(spectrum),"-r",label="Real")
plt.plot(freq,np.imag(spectrum),"-b",label="Imaginary")
#plt.xlim(0,f2+5)
plt.title("Fast fourier transform")
plt.ylabel("Value")
plt.xlabel("Frequency (Hz)")
plt.legend()
plt.grid(True)


plt.show()



In [ ]:
positive_spectrum=spectrum[freq>=0]
positive_freq=freq[freq>=0]

fig=plt.figure(figsize=(20,10))
plt.subplot(2,1,1)

plt.plot(positive_freq,np.abs(positive_spectrum),"-r")
plt.xlim(0,max(freq))
plt.title("Magnitude Fast fourier transform")
plt.ylabel("Value")
plt.xlabel("Frequency (Hz)")
plt.grid(True)

plt.subplot(2,1,2)
plt.plot(positive_freq,np.angle(positive_spectrum),"-r")
plt.xlim(0,max(freq))
plt.title("Phase Fast fourier transform")
plt.ylabel("Value")
plt.xlabel("Frequency (Hz)")
plt.grid(True)

plt.show()



**The decibel (symbol: dB)**  It is used to express the ratio of one value of a power or field quantity to another, on a logarithmic scale, the logarithmic quantity being called the power level or field level, respectively.

It can be used to express a change in value (e.g., +1 dB or −1 dB) or an absolute value. In the latter case, it expresses the ratio of a value to a fixed reference value

In seismic usually we use:

$ dBs = 20* \log{10}{\frac{Value}{Reference}} $


     

In [ ]:
#Printing the fast fourier in dBs

fig=plt.figure(figsize=(20,5))
y=np.abs(positive_spectrum)
plt.plot(positive_freq,20*np.log(y/max(y)),"-r")
plt.xlim(0,max(freq))
plt.title("Fast fourier transform")
plt.ylabel("dBs")
plt.xlabel("Frequency (Hz)")
plt.grid(True)
plt.show()

<a  id="title832"></a> 
<h2>2.2 Periodogram</h2>

In signal processing, a periodogram is an estimate of the spectral density of a signal. 

In [ ]:
from scipy.signal import periodogram


f,Pxx_den=periodogram(signal,float(fs))

fig=plt.figure(figsize=(20,5))
plt.plot(f,Pxx_den,"-r")
plt.xlim(0,max(f))
plt.title("Power spectral density")
plt.ylabel("PSD [N^2/Hz]")
plt.xlabel("Frequency (Hz)")
plt.grid(True)
plt.show()


<a  id="title833"></a> 
<h2>2.3 Wavelet generation in python    </h2> 

Scipy can generate different wavelets for digital processing. Some of the mos used wavelet used are:
    
    * Chirp
    * Gausspulse
    * Sawtooth
    * Sqaure
    * Unit impulse
    * Morlet
    * ricket
    * etc
    

In [ ]:
from scipy.signal import chirp

def ricker(f, dt):
    length=0.5
    t = np.arange(-length/2, (length-dt)/2, dt)
    y = (1.0 - 2.0*(np.pi**2)*(f**2)*(t**2)) * np.exp(-(np.pi**2)*(f**2)*(t**2))
    return t, y

f = 30 
dt=0.004
fig=plt.figure(figsize=(20,5))
plt.subplot(1,2,1)
t, w = ricker(f,dt)
plt.plot(t,w)
plt.grid(True)
plt.title("Ricker wavelet 30Hz")
plt.xlabel("Time(s)")

plt.subplot(1,2,2)
t = np.arange(0, 10, 0.001)
w = chirp(t, f0=6, f1=1, t1=10, method='linear')
plt.plot(t, w)
plt.title("Linear Chirp, f(0)=6, f(10)=1")
plt.xlabel('t (sec)')
plt.grid(True)




<a  id="title834"></a> 
<h2>2.4 Convolution    </h2> 

In mathematics (in particular, functional analysis) convolution is a mathematical operation on two functions (f and g) that produces a third function expressing how the shape of one is modified by the other. The term convolution refers to both the result function and to the process of computing it. It is defined as the integral of the product of the two functions after one is reversed and shifted. And the integral is evaluated for all values of shift, producing the convolution function.

In python there are functions for convolution, FFT convolution and 2D convolution

https://en.wikipedia.org/wiki/Convolution

In [ ]:
import matplotlib.pyplot as plt
from scipy import signal


sig = np.full(300,1)
sig[10]=2
sig[50]=-6
sig[100]=10
sig[150]=-16
sig[200]=-15
sig[250]=2

win = signal.hann(30)
filtered = signal.convolve(sig, win, mode='same') / sum(win)

fig=plt.figure(figsize=(20,10))
plt.subplot(3, 1, 1)
plt.plot(sig)
plt.title('Original pulse')
plt.xlim(0,300)
plt.subplot(3, 1, 2)
plt.plot(win)
plt.title('Filter impulse response')
plt.xlim(0,300)
plt.subplot(3, 1, 3)
plt.plot(filtered)
plt.title('Filtered signal')
plt.xlim(0,300)
#fig.tight_layout()
fig.show()

<a  id="title835"></a> 
<h2>2.5 Correlation    </h2> 

Correlation between signals indicates the measure up to which the given signal resembles another signal.

In other words, if we want to know how much similarity exists between the signals 1 and 2, then we need to find out the correlation of Signal 1 with respect to Signal 2 or vice versa.

https://www.allaboutcircuits.com/technical-articles/understanding-correlation/

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt


sig = np.repeat([0., 1., 1., 0., 1., 0., 0., 1.], 128)
sig_noise = sig + np.random.randn(len(sig))
corr = signal.correlate(sig_noise, np.ones(128), mode='same') / 128

clock = np.arange(64, len(sig), 128)

fig=plt.figure(figsize=(20,10))
plt.subplot(3,1,1)
plt.plot(sig)
plt.plot(clock, sig[clock], 'ro')
plt.title('Original signal')
plt.subplot(3,1,2)
plt.plot(sig_noise)
plt.title('Signal with noise')
plt.subplot(3,1,3)
plt.plot(corr)
plt.plot(clock, corr[clock], 'ro')
plt.axhline(0.5, ls=':')
plt.title('Cross-correlated with rectangular pulse')
plt.margins(0, 0.1)
fig.tight_layout()
fig.show()


<a  id="title836"></a> 
<h2>2.6 Filter design    </h2> 


Infinite Impulse Response (IIR) or Finite Impulse Response (FIR) can designed and applied in python. There are several filters than can be design and applied

https://docs.scipy.org/doc/scipy/reference/signal.html

In [ ]:
import numpy as np
from scipy.signal import butter, lfilter, freqz
import matplotlib.pyplot as plt

In [ ]:

# Filter requirements.
order = 6
fs = 500.0       # sample rate, Hz

cutoff = 50  # desired cutoff frequency of the filter, Hz

nyq = 0.5 * fs
normal_cutoff = cutoff / nyq
b, a = butter(order, normal_cutoff, btype='low', analog=False)

fig=plt.figure(figsize=(20,5))

# Plot the frequency response.
w, h = freqz(b, a, worN=8000)

plt.plot(0.5*fs*w/np.pi, np.abs(h), 'b')
plt.plot(cutoff, 0.5*np.sqrt(2), 'ko')
plt.axvline(cutoff, color='k')
plt.xlim(0, 0.5*fs)
plt.title("Lowpass Filter Frequency Response")
plt.xlabel('Frequency [Hz]')
plt.grid()

In [ ]:

fig=plt.figure(figsize=(20,5))

# First make some data to be filtered.
T = 5.0         # seconds
n = int(T * fs) # total number of samples
#t = np.linspace(0, T, n)
t = np.arange(0,T,(T/n))
# "Noisy" data.  We want to recover the 1.2 Hz signal from this.
data = np.sin(20*2*np.pi*t) + 1.5*np.cos(100*2*np.pi*t) + 0.5*np.sin(120*2*np.pi*t)
plt.plot(t, data, 'b-', label='data')
plt.xlabel('Time [sec]')
plt.xlim(0,1)
plt.grid()
plt.legend()

In [ ]:
#Plot the signal fft


spectrum1=np.fft.fft(data)
freq1 = np.fft.fftfreq(len(data),d=1/fs)
positive_spectrum1=spectrum1[freq1>=0]
positive_freq1=freq1[freq1>=0]

fig=plt.figure(figsize=(20,5))
plt.plot(positive_freq1,np.abs(positive_spectrum1),"-r",label="Original data")

plt.xlim(0,max(freq1))
plt.title("Fast fourier transform")
plt.ylabel("Value")
plt.xlabel("Frequency (Hz)")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
fig=plt.figure(figsize=(20,10))
T = 5.0         # seconds
n = int(T * fs) # total number of samples

#t = np.linspace(0, T, n)
t = np.arange(0,T,(T/n))

# Filter the data, and plot both the original and filtered signals.

# Filter requirements.
order = 6
fs = 500.0       # sample rate, Hz
cutoff = 50  # desired cutoff frequency of the filter, Hz
nyq = 0.5 * fs
normal_cutoff = cutoff / nyq
b, a = butter(order, normal_cutoff, btype='low', analog=False)

y = lfilter(b, a, data)

plt.subplot(2, 1, 2)
plt.plot(t, data, 'b--', label='data')
plt.plot(t, y, 'r-', linewidth=2, label='filtered data')
plt.xlabel('Time [sec]')
plt.xlim(0,1)
plt.grid()
plt.legend()

plt.subplots_adjust(hspace=0.35)
plt.show()

In [ ]:
#Calculate and plot the fft of the data filtered

spectrum2=np.fft.fft(y)
freq2 = np.fft.fftfreq(len(y),d=1/fs)
positive_spectrum2=spectrum2[freq2>=0]
positive_freq2=freq2[freq2>=0]

fig=plt.figure(figsize=(20,5))
plt.plot(positive_freq2,np.abs(positive_spectrum2),"-b",label="Filtered data")
plt.xlim(0,max(freq1))
plt.title("Fast fourier transform")
plt.ylabel("Value")
plt.xlabel("Frequency (Hz)")
plt.grid(True)
plt.legend()
plt.show()